In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import itertools
import re
import matplotlib.pyplot as plt
from collections import Counter
import pickle 

# Steps to generate graphs
### 1. Import all the connectivity matrices/dfs
### 2. Make the matrices square by adding columns of 0s
### 3. Confirm that node order is the same in the row and column of a matrix and that this order is same across all 8 matrices 
### 4. Convert the matrices into weighted and unweighted graphs
### 5. Remove Self-Loops from all the graphs
### 6. Assign specific class and parent class info attributes to all the nodes
### 7. Study special cases: CANL and CANR not having a parent type in the neurontypes excel sheet; the parent class of BWM type is obtained by using serial number at the end
### 8. Remove disconnected nodes

## Importing connectivitiy matrices and creating networks

In [2]:
xls = pd.ExcelFile('data/raw/connectivity_matrices_41586_2021_3778_MOESM4_ESM.xlsx')
df1 = pd.read_excel(xls, 'Dataset1', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)
df2 = pd.read_excel(xls, 'Dataset2', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)
df3 = pd.read_excel(xls, 'Dataset3', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)
df4 = pd.read_excel(xls, 'Dataset4', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)

df5 = pd.read_excel(xls, 'Dataset5', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)
df6 = pd.read_excel(xls, 'Dataset6', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)
df7 = pd.read_excel(xls, 'Dataset7', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)
df8 = pd.read_excel(xls, 'Dataset8', skiprows=[1,3], usecols=[i for i in range(2, 183)], header = 1, index_col=0)

In [3]:
all_c_elegans_dataframes = [df1, df2, df3, df4, df5, df6, df7, df8]

In [4]:
for df in all_c_elegans_dataframes:
    print(df.shape)

(224, 180)
(224, 180)
(224, 180)
(224, 180)
(224, 180)
(224, 180)
(224, 180)
(224, 180)


In [5]:
all([set(df.index).issuperset(set(df.columns)) for df in all_c_elegans_dataframes])

True

In [6]:
df1

,ADFL,ADFR,ADLL,ADLR,AFDL,AFDR,ALML,ALMR,ALNL,ALNR,...,PVNL,PVNR,PVQL,PVQR,RICL,RICR,RID,RIS,RMGL,RMGR
ADFL,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ADFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ADLL,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ADLR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AFDL,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GLRDR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GLRL,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GLRR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GLRVL,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# confirming that the order of columns is the same as that of index
# this must be the case when using a square matrix for creating network
for df in all_c_elegans_dataframes:
    print(all(df.index[:180] == df.columns))

True
True
True
True
True
True
True
True


In [8]:
def square_dataframe(dataframe):
    
    # in all dfs, the columns are less than the rows and are a subset of the rows
    # have to make df square because networkx only accepts square matrices obviously
    # so to do so, I add more columns with zero values
    
    missingnodes = dataframe.index[180:]
    df_to_be_added = pd.DataFrame({k:[0 for i in range(0, 224)] for k in missingnodes})
    
    indices = dataframe.index
    
    dataframe.reset_index(drop=True, inplace=True) # necessary cause otherwise the concatenation happens sideways but with a downward shift because the indices are not aligned
    dataframe_squared = pd.concat([dataframe, df_to_be_added], axis = 1) # adding on the right side of dataframe
    
    dataframe_squared.index = indices # resetting the nodes names as the indices cause they were lost with index was reset above
    
    return dataframe_squared

In [9]:
def create_weighted_graph_from_dataframe(dataframe):
    connectivity_matrix = np.transpose(dataframe.to_numpy()) # TRANSPOSING VERY IMPORTANT cause raw data, 
    #pre neurons are the columns and post neurons are the rows
    
    # number of connections, total weight of all connections, number of self connections
    info = [np.count_nonzero(connectivity_matrix),np.sum(connectivity_matrix), 
            np.sum(np.diagonal(connectivity_matrix))]    
    
    graph = nx.from_numpy_array(connectivity_matrix, create_using=nx.DiGraph)
    
    print(graph)
    print(info)
    
    return graph

In [10]:
df1_squared = square_dataframe(df1)
df2_squared = square_dataframe(df2)
df3_squared = square_dataframe(df3)
df4_squared = square_dataframe(df4)
df5_squared = square_dataframe(df5)
df6_squared = square_dataframe(df6)
df7_squared = square_dataframe(df7)
df8_squared = square_dataframe(df8)

In [11]:
all_c_elegans_dataframes_squared = [df1_squared, df2_squared, df3_squared, df4_squared, df5_squared, df6_squared, df7_squared, df8_squared]

In [12]:
# confirming node order is the same across rows and columns
for df in all_c_elegans_dataframes_squared:
    print(all(df.index == df.columns))

True
True
True
True
True
True
True
True


In [13]:
# confirming that node order is the same in all the datasets: thus, I can use a single relable dictionary
# and nodeid and types will be the same across all the graphs.
for i in range(len(all_c_elegans_dataframes_squared)-1):
    print(all(all_c_elegans_dataframes_squared[i].index == all_c_elegans_dataframes_squared[i+1].index))

True
True
True
True
True
True
True


In [14]:
specific_class_of_nodes_dictionary = {}
for index, specific_class in enumerate(df1_squared.index):
    specific_class_of_nodes_dictionary[index] = specific_class
specific_class_of_nodes_dictionary

{0: 'ADFL',
 1: 'ADFR',
 2: 'ADLL',
 3: 'ADLR',
 4: 'AFDL',
 5: 'AFDR',
 6: 'ALML',
 7: 'ALMR',
 8: 'ALNL',
 9: 'ALNR',
 10: 'AQR',
 11: 'ASEL',
 12: 'ASER',
 13: 'ASGL',
 14: 'ASGR',
 15: 'ASHL',
 16: 'ASHR',
 17: 'ASIL',
 18: 'ASIR',
 19: 'ASJL',
 20: 'ASJR',
 21: 'ASKL',
 22: 'ASKR',
 23: 'AUAL',
 24: 'AUAR',
 25: 'AVM',
 26: 'AWAL',
 27: 'AWAR',
 28: 'AWBL',
 29: 'AWBR',
 30: 'AWCL',
 31: 'AWCR',
 32: 'BAGL',
 33: 'BAGR',
 34: 'DVA',
 35: 'FLPL',
 36: 'FLPR',
 37: 'IL2DL',
 38: 'IL2DR',
 39: 'IL2L',
 40: 'IL2R',
 41: 'IL2VL',
 42: 'IL2VR',
 43: 'OLLL',
 44: 'OLLR',
 45: 'OLQDL',
 46: 'OLQDR',
 47: 'OLQVL',
 48: 'OLQVR',
 49: 'PLNL',
 50: 'PLNR',
 51: 'SAADL',
 52: 'SAADR',
 53: 'SAAVL',
 54: 'SAAVR',
 55: 'SDQL',
 56: 'SDQR',
 57: 'URBL',
 58: 'URBR',
 59: 'URXL',
 60: 'URXR',
 61: 'URYDL',
 62: 'URYDR',
 63: 'URYVL',
 64: 'URYVR',
 65: 'ADAL',
 66: 'ADAR',
 67: 'AIAL',
 68: 'AIAR',
 69: 'AIBL',
 70: 'AIBR',
 71: 'AINL',
 72: 'AINR',
 73: 'AIYL',
 74: 'AIYR',
 75: 'AIZL',
 76: 'AIZ

### dataset7 has self loops

In [15]:
c_elegans_weighted_graph_1 = create_weighted_graph_from_dataframe(df1_squared)
c_elegans_weighted_graph_2 = create_weighted_graph_from_dataframe(df2_squared)
c_elegans_weighted_graph_3 = create_weighted_graph_from_dataframe(df3_squared)
c_elegans_weighted_graph_4 = create_weighted_graph_from_dataframe(df4_squared)
c_elegans_weighted_graph_5 = create_weighted_graph_from_dataframe(df5_squared)
c_elegans_weighted_graph_6 = create_weighted_graph_from_dataframe(df6_squared)
c_elegans_weighted_graph_7 = create_weighted_graph_from_dataframe(df7_squared)
c_elegans_weighted_graph_8 = create_weighted_graph_from_dataframe(df8_squared)

DiGraph with 224 nodes and 775 edges
[np.int64(775), np.int64(1296), np.int64(0)]
DiGraph with 224 nodes and 986 edges
[np.int64(986), np.int64(1895), np.int64(0)]
DiGraph with 224 nodes and 1012 edges
[np.int64(1012), np.int64(2128), np.int64(0)]
DiGraph with 224 nodes and 1136 edges
[np.int64(1136), np.int64(2777), np.int64(0)]
DiGraph with 224 nodes and 1515 edges
[np.int64(1515), np.int64(4116), np.int64(0)]
DiGraph with 224 nodes and 1525 edges
[np.int64(1525), np.int64(4456), np.int64(0)]
DiGraph with 224 nodes and 2202 edges
[np.int64(2202), np.int64(7467), np.int64(12)]
DiGraph with 224 nodes and 2186 edges
[np.int64(2186), np.int64(7970), np.int64(0)]


In [16]:
c_elegans_weighted_graph_7.remove_edges_from(nx.selfloop_edges(c_elegans_weighted_graph_7))

In [17]:
list(nx.selfloop_edges(c_elegans_weighted_graph_7))

[]

In [18]:
all_c_elegans_weighted_graphs = [c_elegans_weighted_graph_1, c_elegans_weighted_graph_2, c_elegans_weighted_graph_3, c_elegans_weighted_graph_4,
                        c_elegans_weighted_graph_5, c_elegans_weighted_graph_6, c_elegans_weighted_graph_7, c_elegans_weighted_graph_8]

In [19]:
for graph in all_c_elegans_weighted_graphs:
    nx.set_node_attributes(graph, specific_class_of_nodes_dictionary, 'specific_class')

In [20]:
neurontypes_df = pd.read_excel('data/raw/neurontypes_41586_2021_3778_MOESM3_ESM.xlsx')
neurontypes_df

,Cell class,Cell type,Classifiers,Vesicle type,Reference (this study if missing),DOI
0,ADA,Interneuron,Most of connections to other neurons and not s...,Mixed clear and dense-core vesicles,NaN,NaN
1,ADE,Modulatory neuron,Dopaminergic,Mixed clear and dense-core vesicles,"Sulston et al., 1975",10.1002/cne.901630207
2,ADF,Sensory neuron,Ciliated sensory dendrites,Mixed clear and dense-core vesicles,"White et al., 1986",10.1098/rstb.1986.0056
3,ADL,Sensory neuron,Ciliated sensory dendrites,Mixed clear and dense-core vesicles,"White et al., 1986",10.1098/rstb.1986.0056
4,AFD,Sensory neuron,Ciliated sensory dendrites,Mixed clear and dense-core vesicles,"White et al., 1986",10.1098/rstb.1986.0056
...,...,...,...,...,...,...
88,BWM06,Muscle,Dendrite-like arms connect to cell bodies with...,No vesicles,NaN,NaN
89,BWM07,Muscle,Dendrite-like arms connect to cell bodies with...,No vesicles,NaN,NaN
90,BWM08,Muscle,Dendrite-like arms connect to cell bodies with...,No vesicles,NaN,NaN
91,GLR,Glia,Wraps neural processes,No vesicles,NaN,NaN


In [21]:
cellclass_info_dict = {}
for row in neurontypes_df.values:
    cellclass = row[0]
    celltype = row[1]
    classifier = row[2]
    vescicle = row[3]
    cellclass_info_dict[cellclass] = {'celltype': celltype, 'classifier': classifier, 'vescicle':vescicle}
    

In [22]:
cellclass_info_dict

{'ADA': {'celltype': 'Interneuron',
  'classifier': 'Most of connections to other neurons and not sensory',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'ADE': {'celltype': 'Modulatory neuron',
  'classifier': 'Dopaminergic',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'ADF': {'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'ADL': {'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'AFD': {'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'AIA': {'celltype': 'Interneuron',
  'classifier': 'Most of connections to other neurons and not sensory',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'AIB': {'celltype': 'Interneuron',
  'classifier': 'Most of connections to other neurons and not sensory',
  'vescicle': 'Mixed cl

In [23]:
with open('data/processed/cellclass_info_dict.pkl', 'wb') as file:
    pickle.dump(cellclass_info_dict, file)

In [24]:
len(cellclass_info_dict)

93

In [25]:
def string_overlap(string1, string2):
    
    larger_string = max([string1, string2], key = len)
    smaller_string = string2 if larger_string == string1 else string1
        
    overlap = 0 
    corresponding_chars = [['_', '_'] for i in range(len(larger_string))]
    for index, char in enumerate(smaller_string):
        corresponding_chars[index][0] = char
    for index, char in enumerate(larger_string):
        corresponding_chars[index][1] = char
    
    for pair in corresponding_chars:
        if pair[0] == pair[1]:
            overlap += 1
        else:
            break
        
    return overlap

In [26]:
string_overlap('ABCDEJFJGF', 'ABCEDEF')

3

In [27]:
# CANL and CANR do not fit in any cell type mentioned in the excel sheet
for nodeid, specific_class in specific_class_of_nodes_dictionary.items():
    if specific_class[:3] not in set(key[:3] for key in cellclass_info_dict.keys()):
        print(specific_class)

CANL
CANR


In [28]:
for node, typ in specific_class_of_nodes_dictionary.items():
    if typ.startswith('CA'):
        print(node)

212
213


In [29]:
# There's only one edge involving a CAN type neuron across all the graphs! 
# The paper on neuropeptidergic connectome says: CAN neurons, which completely lack chemical synapses, show strong and reciprocal neuropeptidergic 
# connectivity with the rest of the nervous system, indicating that this unusual neuron class is well embedded in the neural network
for graph in all_c_elegans_weighted_graphs:
    print(graph.degree(212), graph.degree(213))

0 0
0 0
0 0
0 0
0 0
0 1
0 0
0 0


In [30]:
specific_class_of_nodes_dictionary

{0: 'ADFL',
 1: 'ADFR',
 2: 'ADLL',
 3: 'ADLR',
 4: 'AFDL',
 5: 'AFDR',
 6: 'ALML',
 7: 'ALMR',
 8: 'ALNL',
 9: 'ALNR',
 10: 'AQR',
 11: 'ASEL',
 12: 'ASER',
 13: 'ASGL',
 14: 'ASGR',
 15: 'ASHL',
 16: 'ASHR',
 17: 'ASIL',
 18: 'ASIR',
 19: 'ASJL',
 20: 'ASJR',
 21: 'ASKL',
 22: 'ASKR',
 23: 'AUAL',
 24: 'AUAR',
 25: 'AVM',
 26: 'AWAL',
 27: 'AWAR',
 28: 'AWBL',
 29: 'AWBR',
 30: 'AWCL',
 31: 'AWCR',
 32: 'BAGL',
 33: 'BAGR',
 34: 'DVA',
 35: 'FLPL',
 36: 'FLPR',
 37: 'IL2DL',
 38: 'IL2DR',
 39: 'IL2L',
 40: 'IL2R',
 41: 'IL2VL',
 42: 'IL2VR',
 43: 'OLLL',
 44: 'OLLR',
 45: 'OLQDL',
 46: 'OLQDR',
 47: 'OLQVL',
 48: 'OLQVR',
 49: 'PLNL',
 50: 'PLNR',
 51: 'SAADL',
 52: 'SAADR',
 53: 'SAAVL',
 54: 'SAAVR',
 55: 'SDQL',
 56: 'SDQR',
 57: 'URBL',
 58: 'URBR',
 59: 'URXL',
 60: 'URXR',
 61: 'URYDL',
 62: 'URYDR',
 63: 'URYVL',
 64: 'URYVR',
 65: 'ADAL',
 66: 'ADAR',
 67: 'AIAL',
 68: 'AIAR',
 69: 'AIBL',
 70: 'AIBR',
 71: 'AINL',
 72: 'AINR',
 73: 'AIYL',
 74: 'AIYR',
 75: 'AIZL',
 76: 'AIZ

In [31]:
cellclass_info_dict

{'ADA': {'celltype': 'Interneuron',
  'classifier': 'Most of connections to other neurons and not sensory',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'ADE': {'celltype': 'Modulatory neuron',
  'classifier': 'Dopaminergic',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'ADF': {'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'ADL': {'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'AFD': {'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'AIA': {'celltype': 'Interneuron',
  'classifier': 'Most of connections to other neurons and not sensory',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 'AIB': {'celltype': 'Interneuron',
  'classifier': 'Most of connections to other neurons and not sensory',
  'vescicle': 'Mixed cl

In [32]:
parent_classes_of_nodes = {}

for node_id, nodeclass in specific_class_of_nodes_dictionary.items():
    if nodeclass.startswith('CAN'):
        parent_classes_of_nodes[node_id] = {'parent_class': nodeclass} #No other info available 
        continue
        
    if nodeclass.startswith('BWM'):
        number = nodeclass[-2:]
        parent_class = nodeclass[:3] + number
        parent_classes_of_nodes[node_id] = {'parent_class': parent_class} 
        parent_classes_of_nodes[node_id].update(cellclass_info_dict[parent_class])
        continue
        
    l = []
    for cellclass in cellclass_info_dict:
        l.append((nodeclass, cellclass, string_overlap(nodeclass, cellclass)))
    parent_class = max(l, key=lambda tup:tup[2])[1]
    parent_classes_of_nodes[node_id] = {'parent_class': parent_class} 
    parent_classes_of_nodes[node_id].update(cellclass_info_dict[parent_class])
    

In [34]:
parent_classes_of_nodes

{0: {'parent_class': 'ADF',
  'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 1: {'parent_class': 'ADF',
  'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 2: {'parent_class': 'ADL',
  'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 3: {'parent_class': 'ADL',
  'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 4: {'parent_class': 'AFD',
  'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 5: {'parent_class': 'AFD',
  'celltype': 'Sensory neuron',
  'classifier': 'Ciliated sensory dendrites',
  'vescicle': 'Mixed clear and dense-core vesicles'},
 6: {'parent_class': 'ALM',
  'celltype'

In [35]:
for graph in all_c_elegans_weighted_graphs:
    nx.set_node_attributes(graph, parent_classes_of_nodes, 'parent_class_info')

In [36]:
for graph in all_c_elegans_weighted_graphs:
    print(len(list(nx.isolates(graph))))

37
31
26
21
14
8
3
5


In [37]:
for graph in all_c_elegans_weighted_graphs:
    disconnected_nodes = list(nx.isolates(graph))
    graph.remove_nodes_from(disconnected_nodes)

In [38]:
for graph in all_c_elegans_weighted_graphs:
    print(graph, nx.density(graph))

DiGraph with 187 nodes and 775 edges 0.022281639928698752
DiGraph with 193 nodes and 986 edges 0.026608376511226252
DiGraph with 198 nodes and 1012 edges 0.025944726452340666
DiGraph with 203 nodes and 1136 edges 0.027703262937131153
DiGraph with 210 nodes and 1515 edges 0.034518113465481885
DiGraph with 216 nodes and 1525 edges 0.032838070628768305
DiGraph with 221 nodes and 2191 edges 0.04506375976964212
DiGraph with 219 nodes and 2186 edges 0.045787775962464916


In [39]:
with open('data/processed/all_c_elegans_weighted_graphs.pkl', 'wb') as file:
    pickle.dump(all_c_elegans_weighted_graphs, file)